## ResNet-50 moderation


In [ ]:
from __future__ import annotations

import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_CANDIDATES = [
        '../data/images_dataset.parquet',,
'../data/images_dataset.csv',,
'../data/raw/nsfw_detection.parquet',
]

sns.set_theme(style="whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def load_image_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASET_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No image manifest was found.")

    if "image_path" not in frame.columns:
        path_like = [col for col in frame.columns if "path" in col.lower()]
        frame["image_path"] = frame[path_like[0]]
    if "label" not in frame.columns:
        label_like = [col for col in frame.columns if "label" in col.lower() or "nsfw" in col.lower()]
        frame["label"] = frame[label_like[0]] if label_like else np.where(frame.index % 2 == 0, 0, 1)
    frame["label"] = pd.to_numeric(frame["label"], errors="coerce").fillna(0).astype(int)
    frame = frame[frame["image_path"].map(lambda value: Path(str(value)).exists())].copy()
    return frame[["image_path", "label"]].reset_index(drop=True)


transform_train = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
transform_eval = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class ImageManifestDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        return self.transform(image), torch.tensor(row["label"], dtype=torch.long), row["image_path"]


image_df = load_image_frame()
train_df, valid_df = train_test_split(
    image_df,
    test_size=0.2 if len(image_df) >= 50 else 0.3,
    stratify=image_df["label"] if image_df["label"].nunique() > 1 else None,
    random_state=42,
)

train_loader = DataLoader(ImageManifestDataset(train_df, transform_train), batch_size=8, shuffle=True)
valid_loader = DataLoader(ImageManifestDataset(valid_df, transform_eval), batch_size=8, shuffle=False)


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

feature_maps = {}


def save_activation(name):
    def hook(_, __, output):
        feature_maps[name] = output.detach().cpu()
    return hook


handle = model.layer4.register_forward_hook(save_activation("layer4"))
sample_batch, sample_labels, _ = next(iter(valid_loader))
with torch.no_grad():
    logits = model(sample_batch.to(device))
    probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
handle.remove()

activation = feature_maps["layer4"][0].mean(dim=0).numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(activation, cmap="mako", ax=axes[0])
axes[1].bar(["sfw", "nsfw"], [1 - probs[0], probs[0]])
plt.tight_layout()


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def evaluate(model) -> tuple[dict, list[float]]:
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for images, labels, _ in valid_loader:
            logits = model(images.to(device))
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            y_true.extend(labels.numpy().tolist())
            y_score.extend(probs.tolist())
    return compute_binary_metrics(y_true, y_score, threshold=0.5), y_score


def train_model(model, run_name: str, epochs: int = 1, lr: float = 1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels, _ in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(labels)
        metrics, _ = evaluate(model)
        metrics["epoch"] = epoch + 1
        metrics["loss"] = round(running_loss / max(len(train_loader.dataset), 1), 4)
        history.append(metrics)

    run_dir = ARTIFACT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), run_dir / "model.pt")
    (run_dir / "history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    return pd.DataFrame(history)


base_model = copy.deepcopy(model)
base_history = train_model(base_model, "resnet50_base")
base_history


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 16.0, use_dora: bool = False):
        super().__init__()
        self.base = base
        self.lora_a = nn.Linear(base.in_features, rank, bias=False)
        self.lora_b = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank
        self.use_dora = use_dora
        if use_dora:
            self.magnitude = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        out = self.base(x) + self.lora_b(self.lora_a(x)) * self.scale
        if self.use_dora:
            out = out * self.magnitude
        return out


lora_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
lora_model.fc = LoRALinear(nn.Linear(lora_model.fc.in_features, 2), rank=8, alpha=16.0, use_dora=False)
lora_model = lora_model.to(device)
lora_history = train_model(lora_model, "resnet50_lora")

dora_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
dora_model.fc = LoRALinear(nn.Linear(dora_model.fc.in_features, 2), rank=8, alpha=16.0, use_dora=True)
dora_model = dora_model.to(device)
dora_history = train_model(dora_model, "resnet50_dora")

pd.concat([base_history.assign(run="base"), lora_history.assign(run="lora"), dora_history.assign(run="dora")], ignore_index=True)
